# Adding decision logic to the automation

Import all necessary modules and create a demonstration setup with 4 qubits.

In [ ]:
import numpy as np
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications._automation.workflow.workflow_logic import (
    AdaptFrequencyRange,
    FixedParameterUpdate,
    WorkflowLogic,
)
from laboneq_applications.experiments import (
    qubit_spectroscopy,
)
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

# Create a demonstration QuantumPlatform for a 4-qubit tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=4)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup (1x PQSC, 1x SHFQC, 1x HDAWG):
setup = qt_platform.setup

# And a 4-qubit tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of four LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

# We connect to the session in emulation mode:
session = Session(setup)
session.connect(do_emulation=True)

## Set up a simple automation
Here we will define a basic automation workflow with three layers
performing qubit spectroscopy experiments on four qubits.

First we set up the automation parameters:

In [ ]:
# Define the frequency ranges to be used for the three qubit
# spectroscopy experiments
qs1_qubit_setup = {"frequencies": np.linspace(5.9e9, 6.4e9, 101)}
qs2_qubit_setup = {"frequencies": np.linspace(6.1e9, 6.6e9, 101)}
qs3_qubit_setup = {"frequencies": np.linspace(5.8e9, 6.2e9, 101)}

# For simplicity we will use the same options for each layer
options = {
    "evaluate": True,
    "update": True,
    "count": 2048,
    "active_reset": True,
}

qs1_parameters = {q: dict(qs1_qubit_setup) for q in ["q0", "q1", "q2", "q3"]}
qs1_parameters["options"] = options
qs2_parameters = {q: dict(qs2_qubit_setup) for q in ["q0", "q1", "q2", "q3"]}
qs2_parameters["options"] = options
qs3_parameters = {q: dict(qs3_qubit_setup) for q in ["q0", "q1", "q2", "q3"]}
qs3_parameters["options"] = options

# Combine the parameters for each layer into one dictionary
# that will be passed to the automation
automation_parameters = {
    "qs1": qs1_parameters,
    "qs2": qs2_parameters,
    "qs3": qs3_parameters,
}

We can now set up our automation workflow. The three qubit spectroscopy experiments will be
executed one after the other. Using `auto.plot()` we can verify the dependencies between layers.

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=automation_parameters,
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs1",
    depends_on=["__root__"],
)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs2",
    depends_on=["qs1"],
)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2"],
    key="qs3",
    depends_on=["qs2"],
)

auto.add_layer(qs1)
auto.add_layer(qs2)
auto.add_layer(qs3)
auto.plot();

Once we have verified that we have our desired automation graph we can run it.

In [ ]:
auto.run()
auto.plot();

In this case, the execution of each experiment is a success and we have not implemented
any decision logic yet.
Let's quickly inspect the second layer to see the parameters relevant for the next section.

In [ ]:
qs2 = auto.get_layer("qs2")
f"Layer {qs2.key} status: {qs2.status.value}, failed {qs2.fail_count} times and succeeded {qs2.success_count} times."

This shows us that the layer passed on the first try and it didn't fail.
Next we will examine the behavior of the framework with a failing node.

## Run a simple automation with a failing node
We have already set up our `automation_parameters`. To make a failing node
we will cheat a little bit and set the `evaluation_fit_r2_thresholds` of
`q2` in layer `qs2` to `1.0`.

In [ ]:
# Force the node to fail by placing a high threshold for the $r^2$ of the fit
# in the evaluate_experiment task
qs2 = auto.get_layer("qs2")
qs2.workflow_parameters["q2"]["evaluation_fit_r2_thresholds"] = 1.0

Now we can reset the automation, reload the `automation_parameters`
and rerun our automation.

In [ ]:
auto.reset()

auto.run()

auto.plot()

What happened here is that `q2` of layer `qs2` was failing as it's $r^2$ was
below the evaluation threshold of 1.0 that we set. Each fail increments the 
`fail_count` of the layer until it reaches a `max_fail_count` (default is equal to 
4, but it can be changed by `qs2.max_fail_count = 4`, for example).
Reaching the max fail count, the node is disabled and the automation continues to the
next layer.

Let's inspect the status of `qs2_q2` to confirm this:

In [ ]:
qs2 = auto.get_layer("qs2")
print(
    f"Layer status: {qs2.status.value}, failed {qs2.fail_count} times and succeeded {qs2.success_count} times."
)
for n in qs2.nodes:
    print(
        f"node {n.key} status: {n.status.value}, failed {n.fail_count} and succeeded {n.success_count} times."
    )

## Automation workflow with decision logic
So far we saw a straightforward run of the automation with sequential execution of layers. 
We demonstrated the possibility to 'rerun until success'. Now we can complicate things by introducing
extra logic that changes the layer parameters upon rerun.

Let's first define our decision logic parameters and then apply them to the automation.

In [ ]:
logic = FixedParameterUpdate(
    new_layer_key="qs2",
    parameter_changes={
        "q2": {
            "evaluation_fit_r2_thresholds": -0.0003,
        },
    },
    relative=False,
)
qs2 = auto.get_layer("qs2")
qs2.workflow_parameters["q2"]["evaluation_fit_r2_thresholds"] = 1.0
qs2.logic = logic

The `FixedParameterUpdate` is an example implementation of a simple decision logic.
It is a class derived from `WorkflowLogic` and it updates automation parameters upon rerun of a failed
layer. The parameters can be changed by a fixed value (in this example we subtract `-0.0003` from
`evaluation_fit_r2_thresholds` on every rerun). We can also set `relative` to `True` which will 
change the value by a given percentage (e.g. `-0.1` would correspond to a decrease of 10%).

It has a method `run_executable` that takes a `WorkflowLayer` as input, and
outputs a `tuple[str, dict]` with the key of the next layer to be executed and a
(sub-)dictionary of the new parameters to be applied to the layer. The `new_layer_key`
in our `logic` dictionary gives the next layer to be executed upon failing. In
this example we ask the automation to rerun the layer, but we can jump to any
previous layer. The `parameter_changes` sets which parameter to change
(`evaluation_fit_r2_thresholds` in this example) and on which qubit (`q2`).

Let's set up the automation, set the new parameters and rerun it to see what happens.

In [ ]:
auto.reset()
auto.run()
auto.plot()

In [ ]:
qs2 = auto.get_layer("qs2")
print(
    f"Layer status: {qs2.status.value}, failed {qs2.fail_count} times and succeeded {qs2.success_count} times."
)
for n in qs2.nodes:
    print(
        f"node {n.key} status: {n.status.value},\tfailed {n.fail_count} and succeeded {n.success_count} times."
    )

In this example we rerun the same layer upon failure. On every attempt, the decision logic changed the `evaluation_fit_r2_thresholds` by `-0.0003`. In this demo $r^2 \approxeq 0.9991$, so on the third attempt to rerun the layer $r^2 > $ `evaluation_fit_r2_thresholds` and the layer passes.

Note that here we just rerun the layer but it is possible to jump to any previous layer by changing the `new_layer_key` argument of our `FixedParameterUpdate` class.

In [ ]:
logic = FixedParameterUpdate(
    # We jump to qs1 if qs2 fails
    new_layer_key="qs1",
    parameter_changes={
        "q2": {
            "evaluation_fit_r2_thresholds": -0.0003,
        },
    },
    relative=False,
)
qs2 = auto.get_layer("qs2")
qs2.workflow_parameters["q2"]["evaluation_fit_r2_thresholds"] = 1.0
qs2.logic = logic

auto.reset()
auto.run()

In [ ]:
print(f"qs1 failed {qs1.fail_count} times and succeeded {qs1.success_count} times.")
print(f"qs2 failed {qs2.fail_count} times and succeeded {qs2.success_count} times.")
print(f"qs3 failed {qs3.fail_count} times and succeeded {qs3.success_count} times.")

##  Another example of decision logic
Another example decision logic is the `AdaptFrequencyRange`, which adapts the workflow parameters based on the results of the previous run. We can rerun a layer for a number of `iterations`. With `new_layer_key` we specify where to jump after executing the layer.
The dictionary `range_thresholds` specifies what multiplier to apply to a given frequency range. In this example, if the frequency range is between 0 and 200 MHz, increase the range by 1.1 times. If the range is between 200 MHz and 400 MHz, increase by 1.2 times and so on.

Here we set up the logic, check the initial frequency range, run the automation, and check the frequency range at the end.

In [ ]:
logic = AdaptFrequencyRange(
    new_layer_key="qs2",
    range_thresholds={
        0: 1.1,
        200e6: 1.2,
        400e6: 1.3,
        600e6: 1.4,
    },
    iterations=3,
)
qs2 = auto.get_layer("qs2")
qs2.logic = logic
frequencies = qs2.workflow_parameters["q0"]["frequencies"]
f"Frequency range: {int((max(frequencies) - min(frequencies)) / 1e6)} MHz"

In [ ]:
auto.reset()
auto.run()
auto.plot()

In [ ]:
frequencies = qs2.workflow_parameters["q0"]["frequencies"]
f"Frequency range: {int((max(frequencies) - min(frequencies)) / 1e6)} MHz"

Note here that the frequency range was 500 MHz and it was in the bracket 400-600 MHz with multiplier 1.3. After the first iteration it went to 650 MHz and jumped to the next bracket with multiplier 1.4.

## Custom decision logic

Here we will define our own custom logic to get a better feeling of the internals.
Our decision logic class is derived from `WorkflowLogic` and has to have a `run_executable` method that takes a `WorkflowLayer` and returns the new layer key and new workflow parameters.

In [ ]:
class ScanFrequencies(WorkflowLogic):
    def __init__(
        self,
        new_layer_key: str,
        frequency_offset: float,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.new_layer_key = new_layer_key
        self.frequency_offset = frequency_offset

    def run_executable(self, layer: WorkflowLayer):
        """Return next layer and new workflow parameters."""
        new_params = {}
        for q in layer.quantum_elements:
            new_params[q] = {}

            # Get the used frequencies from the workflow results
            frequencies = layer.workflow_results[0].output.data[q].result.axis[0][0]
            # Apply offset
            frequencies += self.frequency_offset
            new_params[q]["frequencies"] = frequencies

        return self.new_layer_key, new_params


logic = ScanFrequencies(new_layer_key="qs2", iterations=3, frequency_offset=100e6)
qs2.workflow_parameters = {
    q: {"frequencies": np.linspace(6e9, 6.1e9)} for q in ["q0", "q1", "q2"]
}
qs2.logic = logic

In [ ]:
# Check the initial frequency range of qs2
frequency_range = qs2.workflow_parameters["q0"]["frequencies"]
f"Frequency range from {int(min(frequency_range) / 1e6)} MHz to {int(max(frequency_range) / 1e6)} MHz"

In [ ]:
# Reset the automation and run it again
auto.reset()
auto.run()

In [ ]:
# Check the final frequency range of qs2
frequency_range = qs2.workflow_parameters["q0"]["frequencies"]
f"Frequency range from {int(min(frequency_range) / 1e6)} MHz to {int(max(frequency_range) / 1e6)} MHz"